# Partial states: silence, transitions, and the beam as a process

One run of utter's `scripts/partial_states.py`: built streams of Speech Commands words with chosen pauses and finishes cut from the background recordings, decoded at 40 ms blocks with eight readings and partial words on. Tables: `streams`, `words` (each word's position and energy onset/offset in the stream), `gaps` (pause or finish, where, how long, from which recording), `finals` (where the decoder ended an utterance), `stable` (rank 0's hold per block), and `advances` (one row per decoder advance and reading: lead, velocity, age, relation to rank 0, and the trailing `[sil]` span on the best path).

The first figure is the one the tables exist for: a whole stream as a process, every reading's standing over time against what was actually said.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RUN = Path("../data/2026-09-12-states")
manifest = json.loads((RUN / "MANIFEST.json").read_text())
T = {name: pd.read_parquet(RUN / f"{name}.parquet") for name in ("streams", "words", "gaps", "finals", "stable", "advances")}
adv, words, gaps, finals, stable = (T[k] for k in ("advances", "words", "gaps", "finals", "stable"))
test = lambda df: df[df.split == "testing"]
MS = 16
REL = {"same": "#2a78d6", "extends": "#eb6834", "differs": "#1baf7a", "prefix": "#4a3aa7"}   # categorical slots 1, 2, 3, 7
print(json.dumps({k: manifest[k] for k in ("exported", "harness", "harness_args", "wheel", "utter_head", "utter_dirty")}, indent=1))
T["streams"].groupby("split").agg(streams=("key", "size"), minutes=("samples", lambda s: round(s.sum() / 16000 / 60, 1)), words=("n_words", "sum"), finals=("n_finals", "sum"), advances=("n_advances", "sum"))

## One stream as a process

Top lane, `said`: one bar per word from its energy onset to offset. `in partial`: the diamond is the advance from which rank 0's partial holds that word in its place for the rest of the utterance, the earliest a host lifting it would have been right; the dotted lead-in runs from the word's onset, so its length is the wait. A red cross marks a word rank 0 never held. Hover either for the exact ms. Each word's span is shaded down through every panel below, so pauses and finishes are the unshaded gaps. Dashed verticals are the decoder's finals. Second: every reading that ever stood in the top three, its lead over the best other reading at each advance, coloured by its relation to rank 0 at that advance. Third: the trailing `[sil]` span on the best path, the end-of-speech clock, and rank 0's hold. Bottom: every 40 ms block a host would have called `PartialResult()` on — light ticks where the returned partial repeated the last one, coloured ticks where it changed (orange when rank 0's hypothesis grew a word, blue otherwise). Zoom into any transition.

In [ ]:
BLOCK_MS = 40   # scripts/partial_states.py's block interval for this run; a live host gets PartialResult() after every one of these, whether or not its content changed

def heard(key):
    top = adv[(adv.key == key) & (adv["rank"] == 0)].sort_values("adv")
    # a word rank 0 ends its segment with, and the first advance from which it holds that place to the end: the earliest a host lifting it would have been right
    held = []
    for _, s in top.groupby("seg", sort=False):
        toks = [[] if t == "[sil]" else t.split() for t in s.text]; ms = s.ms.to_numpy()
        for i, tok in enumerate(toks[-1]):
            j = len(toks)
            while j > 0 and len(toks[j - 1]) > i and toks[j - 1][i] == tok: j -= 1
            held.append((tok, ms[j]))
    # segments are the decoder's, not the built utterances, so a said word takes the first unclaimed same-label hold between its onset and the next word's offset
    w = words[words.key == key].sort_values("onset")
    on, off = (w.onset / MS).to_numpy(), (w.offset / MS).to_numpy()
    claimed, out = set(), []
    for n, label in enumerate(w.label):
        bound = off[n + 1] if n + 1 < len(w) else np.inf
        m = next((c for c, (tok, h) in enumerate(held) if c not in claimed and tok == label and on[n] <= h < bound), None)
        if m is not None: claimed.add(m)
        out.append(np.nan if m is None else held[m][1])
    return w.assign(on_ms=on, off_ms=off, held_ms=out)

def stream_view(key, start_ms=None, end_ms=None):
    a_full = adv[adv.key == key]
    lo = (start_ms if start_ms is not None else 0.0) // BLOCK_MS * BLOCK_MS
    hi = end_ms if end_ms is not None else float(a_full.ms.max())
    a = a_full[(a_full.ms >= lo) & (a_full.ms <= hi)]
    w = heard(key); w = w[(w.off_ms >= lo) & (w.on_ms <= hi)]
    f = finals[(finals.key == key) & (finals.fed / MS >= lo) & (finals.fed / MS <= hi)]; st = stable[stable.key == key]
    shown = set(a[a["rank"] <= 2].text)
    top = a[a["rank"] == 0].sort_values("adv")
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, row_heights=[0.16, 0.48, 0.22, 0.14], vertical_spacing=0.025)

    on, off, got = w.on_ms.to_numpy(), w.off_ms.to_numpy(), w.held_ms.to_numpy()
    for x0, x1 in zip(on, off):
        fig.add_vrect(x0=x0, x1=x1, fillcolor="#2a78d6", opacity=0.12, line_width=0, row="all", col=1, exclude_empty_subplots=False)
    gap = lambda x: np.column_stack([x, np.full(len(x), np.nan)]).ravel()
    fig.add_trace(go.Scatter(x=gap(np.column_stack([on, off])), y=gap(np.ones((len(w), 2))), mode="lines", line=dict(color="#2a78d6", width=10), showlegend=False,
                             customdata=np.repeat(np.column_stack([w.label, on.round(), off.round()]), 3, axis=0),
                             hovertemplate="said: %{customdata[0]}<br>%{customdata[1]}-%{customdata[2]} ms<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=(on + off) / 2, y=np.where(np.arange(len(w)) % 2, 2.35, 1.7), mode="text", text=w.label, textfont=dict(size=12, color="#1f1f1d"),
                             showlegend=False, hoverinfo="skip"), row=1, col=1)
    ok = ~np.isnan(got)
    fig.add_trace(go.Scatter(x=gap(np.column_stack([on[ok], got[ok]])), y=gap(np.zeros((ok.sum(), 2))), mode="lines", line=dict(color="#898781", width=1.5, dash="dot"),
                             showlegend=False, hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=got[ok], y=np.zeros(ok.sum()), mode="markers", marker=dict(symbol="diamond", size=11, color="#1f1f1d"), showlegend=False,
                             customdata=np.column_stack([w.label[ok], (got - on)[ok].round(), (got - off)[ok].round()]),
                             hovertemplate="in the partial from %{x:.0f} ms: %{customdata[0]}<br>%{customdata[1]} ms after it started, %{customdata[2]:+} ms against its end<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=off[~ok], y=np.zeros((~ok).sum()), mode="markers", marker=dict(symbol="x", size=11, color="#d03b3b"), showlegend=False,
                             text=w.label[~ok], hovertemplate="%{text}: never held by rank 0's partial<extra></extra>"), row=1, col=1)

    for _, ff in f.iterrows():
        fig.add_vline(x=ff.fed / MS, line=dict(color="#898781", dash="dash", width=1), row="all", col=1, exclude_empty_subplots=False)
    for text, s in a[a.text.isin(shown)].groupby("text", sort=False):
        s = s.sort_values("adv")
        fig.add_trace(go.Scatter(x=s.ms, y=s.lead, mode="lines+markers", name=text, line=dict(width=1.5),
                                 marker=dict(size=6, color=[REL[r] for r in s.relation]),
                                 customdata=np.stack([s.lead_delta.round(2), s.relation, s.age_ms], axis=1),
                                 hovertemplate=f"{text}<br>%{{x:.0f}} ms<br>lead %{{y:.2f}}<br>velocity %{{customdata[0]}}<br>%{{customdata[1]}}, age %{{customdata[2]}} ms<extra></extra>"),
                      row=2, col=1)
    fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1), row=2, col=1)
    fig.add_trace(go.Scatter(x=top.ms, y=top.sil_span_ms, mode="lines", name="trailing [sil] span (ms)", line=dict(color="#898781", width=2)), row=3, col=1)
    sel = st[(st.fed / MS >= lo) & (st.fed / MS <= hi)].sort_values("fed")
    if len(sel):
        fig.add_trace(go.Scatter(x=sel.fed / MS, y=sel.stable_ms, mode="lines", name="rank 0 stable_ms", line=dict(color="#eb6834", width=1.5)), row=3, col=1)

    # a partial fires on every block regardless of content (utter's stream loop calls PartialResult()
    # unconditionally); `advances` rows only mark the blocks where what it returned actually changed
    grid = np.arange(lo, hi + BLOCK_MS, BLOCK_MS)
    repeated = grid[~np.isin(np.round(grid).astype(int), np.round(top.ms).astype(int))]
    fig.add_trace(go.Scatter(x=repeated, y=np.zeros(len(repeated)), mode="markers", showlegend=False, hoverinfo="skip",
                             marker=dict(symbol="line-ns", size=7, line=dict(width=1, color="#dedcd2"))), row=4, col=1)
    fig.add_trace(go.Scatter(x=top.ms, y=np.zeros(len(top)), mode="markers", showlegend=False,
                             marker=dict(symbol="line-ns", size=11, line=dict(width=2, color=["#eb6834" if gr else "#2a78d6" for gr in top.grew])),
                             customdata=np.stack([top.text, top.grew], axis=1),
                             hovertemplate="partial changed at %{x:.0f} ms<br>top: %{customdata[0]}<br>grew: %{customdata[1]}<extra></extra>"), row=4, col=1)

    fig.update_layout(template="plotly_white", height=860, title=f"{key}: marker colour = relation to rank 0 (blue same, orange extends, green differs, violet prefix)",
                      legend=dict(orientation="h", y=-0.05), hovermode="closest")
    fig.update_xaxes(range=[lo, hi])
    fig.update_yaxes(range=[-0.6, 2.8], tickvals=[0, 1], ticktext=["in partial", "said"], showgrid=False, zeroline=False, row=1, col=1)
    fig.update_yaxes(title_text="lead over the best<br>other reading (nats)", row=2, col=1)
    fig.update_yaxes(title_text="ms", row=3, col=1)
    fig.update_yaxes(visible=False, range=[-1, 1], row=4, col=1)
    fig.update_xaxes(title_text="ms of audio fed", row=4, col=1)
    return fig

stream_view("testing/0", 0, 25000)

## Kept silence: the hover

Inside finish gaps, a second after the word before (past the endpoint's restart) and 480 ms before the word after, and on the recordings alone: what rank 0 is, how far it leads, and how fast that lead moves. The hypothesis was that silence is kept as a hovering lead, not a growing one.

In [ ]:
a = test(adv); w = test(words); g = test(gaps)
top = a[a["rank"] == 0].copy()
# a finish-gap interior: after the gap's word offset + 480 ms and before the next onset - 480 ms
interior = []
for key, gg in g[g.kind == "finish"].groupby("key"):
    ww = w[w.key == key].sort_values("pos")
    for _, row in gg.iterrows():
        prev = ww[ww.pos < row.pos].tail(1); nxt = ww[ww.pos > row.pos].head(1)   # words ordered by position; a gap follows the word before it
        lo = (prev.offset.iloc[0] if len(prev) else row.pos) / MS + 1000   # past rule2's restart and the phantoms of the first second
        hi = (nxt.onset.iloc[0] / MS - 480) if len(nxt) else np.inf
        sel = top[(top.key == key) & (top.ms >= lo) & (top.ms <= hi)]
        interior.append(sel)
interior = pd.concat(interior)
print(f"finish-gap interiors (1000 ms after the offset, 480 ms before the onset): {len(interior)} advances; rank 0 is [sil] on {(interior.text == '[sil]').mean():.1%}")
sil = interior[interior.text == "[sil]"]
print(f"[sil] lead: mean {sil.lead.mean():.2f} sd {sil.lead.std():.2f}; velocity mean {sil.lead_delta.mean():+.3f}, p50 {sil.lead_delta.median():+.3f}")
fig = make_subplots(rows=1, cols=2, subplot_titles=("[sil] lead in kept silence (nats)", "[sil] velocity in kept silence (nats per advance)"))
fig.add_trace(go.Histogram(x=sil.lead, nbinsx=60, marker_color="#2a78d6", name="lead"), row=1, col=1)
fig.add_trace(go.Histogram(x=sil.lead_delta.dropna(), nbinsx=80, marker_color="#eb6834", name="velocity"), row=1, col=2)
fig.update_layout(template="plotly_white", height=380, showlegend=False, bargap=0.05)
fig

## Transition-aligned velocity

Every word onset in the testing streams aligned at advance 0 (the first advance after the energy onset): the mean velocity of the `[sil]` reading while it leads, of whatever leads, and of the best `extends` reading, from three advances before to two after. The question is whether anything moves *before* the onset.

In [ ]:
rows = []
a_sorted = a.sort_values(["key", "adv"])
for key, ww in w.groupby("key"):
    ak = a_sorted[a_sorted.key == key]
    advs = ak.drop_duplicates("adv")[["adv", "fed"]].reset_index(drop=True)
    for _, word in ww.iterrows():
        k0 = advs.index[advs.fed >= word.onset]
        if not len(k0): continue
        k0 = int(k0[0])
        for off in range(-3, 3):
            k = k0 + off
            if k < 0 or k >= len(advs): continue
            at = ak[ak.adv == advs.adv[k]]
            lead0 = at[at["rank"] == 0]
            silr = at[(at.text == "[sil]") & (at["rank"] == 0)]
            ext = at[at.relation == "extends"].sort_values("rank").head(1)
            rows.append(dict(offset=off, series="[sil] while leading", v=silr.lead_delta.iloc[0] if len(silr) else np.nan))
            rows.append(dict(offset=off, series="rank 0", v=lead0.lead_delta.iloc[0] if len(lead0) else np.nan))
            rows.append(dict(offset=off, series="best extends", v=ext.lead_delta.iloc[0] if len(ext) else np.nan))
al = pd.DataFrame(rows)
summ = al.groupby(["series", "offset"]).v.agg(mean="mean", q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75), n="count").reset_index()
print(summ.pivot(index="offset", columns="series", values="mean").round(2))
fig = go.Figure()
cols = {"[sil] while leading": "#4a3aa7", "rank 0": "#2a78d6", "best extends": "#eb6834"}
for name, s in summ.groupby("series"):
    fig.add_trace(go.Scatter(x=s.offset, y=s.q75, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=s.offset, y=s.q25, mode="lines", line=dict(width=0), fill="tonexty", fillcolor=cols[name] + "22", showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=s.offset, y=s["mean"], mode="lines+markers", name=name, line=dict(color=cols[name], width=2.5), marker=dict(size=9)))
fig.add_vline(x=0, line=dict(color="#898781", dash="dash")); fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1))
fig.update_layout(template="plotly_white", height=460, xaxis=dict(title="advances from the word's energy onset", tickvals=list(range(-3, 3))),
                  yaxis_title="velocity, mean and interquartile band (nats per advance)")
fig

## Momentum by relation

Consecutive velocities of the same reading, 1-4 nats from the lead, split by what the reading is to rank 0. The empty reading's lead persists; a rival's reverts.

In [ ]:
s = a.sort_values(["key", "text", "adv"]).copy()
grp = s.groupby(["key", "text"])
s["next_delta"] = grp.lead_delta.shift(-1); s["next_adv"] = grp.adv.shift(-1)
pairs = s[(s.next_adv == s.adv + 1) & s.lead_delta.notna() & s.next_delta.notna() & s.lead.notna()].copy()
pairs["band"] = pd.cut(pairs.lead.abs(), [0, 1, 4, np.inf], labels=["<1", "1-4", ">=4"], right=False)
band = pairs[pairs.band == "1-4"]
print(f"1-4 nat band, all readings: n={len(band)} r={band.lead_delta.corr(band.next_delta):+.3f}")
for rel, gg in band.groupby("relation"):
    print(f"  {rel:8s}: n={len(gg):6d} r={gg.lead_delta.corr(gg.next_delta):+.3f}")
fig = go.Figure()
for rel in ("same", "extends", "differs", "prefix"):
    gg = band[band.relation == rel]
    fig.add_trace(go.Scattergl(x=gg.lead_delta, y=gg.next_delta, mode="markers", name=f"{rel} (n={len(gg)})",
                               marker=dict(color=REL[rel], size=5, opacity=0.45), text=gg.key + "  " + gg["text"],
                               hovertemplate="%{text}<br>now %{x:.2f}<br>next %{y:.2f}<extra></extra>"))
lim = float(np.nanpercentile(np.abs(band[["lead_delta", "next_delta"]].values), 99.5))
fig.add_shape(type="line", x0=-lim, y0=-lim, x1=lim, y1=lim, line=dict(color="#c3c2b7", dash="dot"))
fig.update_layout(template="plotly_white", height=560, xaxis=dict(title="velocity at advance k (nats)", range=[-lim, lim]),
                  yaxis=dict(title="velocity at advance k+1 (nats)", range=[-lim, lim]))
fig

## The end-of-speech trade

For every silent stretch after a word, from its energy offset to the next word's energy onset, the trailing `[sil]` span at each advance against whether the stretch was a pause (another word came) or a finish. A bound on the span calls a finish; the curve is what each bound mistakes and catches. The built pause lengths are uniform 100-800 ms, which drives these numbers; TD-8's private figure comes from real pauses.

In [ ]:
rows = []
for key, gg in g.groupby("key"):
    ww = w[w.key == key].sort_values("pos"); tk = top[top.key == key].sort_values("adv")
    for _, row in gg.iterrows():
        prev = ww[ww.pos < row.pos].tail(1)
        if not len(prev): continue
        nxt = ww[ww.pos > row.pos].head(1)
        start = prev.offset.iloc[0]; end = nxt.onset.iloc[0] if len(nxt) else row.pos + row.samples   # to the next word's energy onset
        sel = tk[(tk.fed >= start) & (tk.fed <= end)]
        rows.append(dict(kind=row.kind, key=key, gap_ms=row.samples / MS, max_span=sel.sil_span_ms.max() if len(sel) else 0))
sp = pd.DataFrame(rows)
bounds = np.arange(100, 1300, 50)
curve = pd.DataFrame({"bound_ms": bounds,
                      "pauses mistaken": [(sp[sp.kind == "pause"].max_span >= b).mean() for b in bounds],
                      "finishes called": [(sp[sp.kind == "finish"].max_span >= b).mean() for b in bounds]})
fig = go.Figure()
fig.add_trace(go.Scatter(x=curve.bound_ms, y=curve["finishes called"], name="finishes called", mode="lines+markers", line=dict(color="#2a78d6", width=2.5)))
fig.add_trace(go.Scatter(x=curve.bound_ms, y=curve["pauses mistaken"], name="pauses mistaken for a finish", mode="lines+markers", line=dict(color="#eb6834", width=2.5)))
for b, lab in ((300, "TD-8's 300 ms"), (500, "fitted 500 ms")):
    fig.add_vline(x=b, line=dict(color="#898781", dash="dash"), annotation_text=lab, annotation_position="top")
fig.update_layout(template="plotly_white", height=440, xaxis_title="bound on the trailing [sil] span (ms)", yaxis=dict(tickformat=".0%", title="share"))
print(f"pauses {int((sp.kind=='pause').sum())}, finishes {int((sp.kind=='finish').sum())}; at 300 ms: {curve.loc[curve.bound_ms==300, 'pauses mistaken'].iloc[0]:.1%} mistaken, {curve.loc[curve.bound_ms==300, 'finishes called'].iloc[0]:.1%} called")
fig